In [2]:
# Imports
from datasets import load_from_disk
from ml4setk.Parsing.Code.TreeSitterQuery import TreeSitterQuery
import re
import SATDDetector

language = 'Java'

# Load local dataset
ds = load_from_disk("../../data/Java/satd-10k")
print(len(ds))

def preprocess_comments(comments):
    """
    Preprocess comments to:
    1. Merge consecutive single-line comments into a single block.
    2. Exclude license comments at the beginning of the file that do not contain task annotations.
    3. Exclude commented-out code.
    4. Exclude Javadoc comments that do not contain task annotations.
    """
    processed_comments = []

    # Rule 1: Merge consecutive single-line comments
    # comments = merge_single_line_comments(comments)

    # Rule 2: Exclude license comments at the beginning of the file
    if comments and is_license_comment(comments[0][2]) and not contains_task_annotation(comments[0][2]):
        comments = comments[1:]  # Remove the first comment if it's a license comment

    for prefix, suffix, comment in comments:
        has_task_annotation = contains_task_annotation(comment)

        # Rule 3: Exclude commented-out code
        if is_commented_out_code(comment) and not has_task_annotation:
            continue

        # Rule 4: Exclude Javadoc comments without task annotations
        if is_javadoc_comment(comment) and not has_task_annotation:
            continue

        # Add the processed comment to the list
        processed_comments.append((prefix, suffix, comment))

    return processed_comments


TASK_PATTERN = re.compile(r'\b(?:todo|fixme|xxx)\b', re.IGNORECASE)
LICENSE_PATTERN = re.compile(r'\b(?:copyright|license|all rights reserved|permission|redistribution)\b', re.IGNORECASE)
SOURCE_CODE_REGEX = re.compile(
    r"else\s*\{|" +
    r"try\s*\{|" +
    r"do\s*\{|" +
    r"finally\s*\{|" +
    r"if\s*\(|" +
    r"for\s*\(|" +
    r"while\s*\(|" +
    r"switch\s*\(|" +
    r"(?:Long|Byte|Double|Float|Integer|Short|BigDecimal|BigInteger|Character|Boolean|String)\s*\(|" +
    r"assert\s*\(|" +
    r"System\.out\.|" +
    r"public\s+void|" +
    r"private\s+static\s+final|" +
    r"catch\s*\("
)

def contains_task_annotation(comment):
    return bool(TASK_PATTERN.search(comment))

def is_license_comment(comment):
    return bool(LICENSE_PATTERN.search(comment))

def is_commented_out_code(comment):
    return bool(SOURCE_CODE_REGEX.search(comment))

def is_javadoc_comment(comment):
    return comment.lstrip().startswith("/**")

def encode_bitmask_string(code_len: int, spans: list[tuple[int, int]]) -> str:
    mask = ['0'] * code_len
    for start, end in spans:
        mask[start:end] = ['1'] * (end - start)
    return ''.join(mask)

def decode_bitmask_string(mask: str) -> list[tuple[int, int]]:
    spans = []
    start = None
    for i, bit in enumerate(mask):
        if bit == '1':
            if start is None:
                start = i
        else:
            if start is not None:
                spans.append((start, i))
                start = None
    if start is not None:
        spans.append((start, len(mask)))
    return spans

10000


In [3]:
# Function to classify SATD for a batch of rows
def classify_satd(batch):
    count = 0
    java_tree_query = TreeSitterQuery('java')
    detector = SATDDetector.SATDDetector()
    batch_bitmasks = []
    print("Starting SATD classification...")

    for i, (filename, content) in enumerate(zip(batch['file_name'], batch["content"])):
        print(f"Processing file {i + 1}/{len(batch['file_name'])}: {filename}")
        spans = []
        comments = java_tree_query.parse(content, '(line_comment) @comment (block_comment) @comment')
        preprocessed_comments = preprocess_comments(comments)
        for (prefix, suffix, comment) in preprocessed_comments:
            count += 1
            if count % 1000 == 0:
                print(f"Processing comment # {count}: {comment} in file {filename}")
            if comment == None:
                print("Empty comment found, skipping...")
                continue
            clean_comment = " ".join(comment.split())
            if detector.classify(clean_comment) == ">SATD":
                start = prefix
                end = len(content) - suffix
                spans.append((start, end))
                print(f"Found SATD: {clean_comment} at {start}:{end}")

        batch_bitmasks.append(encode_bitmask_string(len(content), spans))

    detector.close()
    return {"bitmask_satd": batch_bitmasks}


ds_satd = ds.select(range(8000)).map(
    classify_satd,
    batched=True,
    batch_size=1000,
)

# Save the dataset with bitmask
# ds_satd.save_to_disk(f"/scratch/lcwitte/satd-annotate/{language.lower()}/satd-all")

# count SATD
satd_count = 0
for file in ds_satd:
    spans = decode_bitmask_string(file["bitmask_satd"])
    satd_count += len(spans)
print(f"Total SATD comments: {satd_count}")


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

c:\Users\lucaw\TUProjects\RP\ML4SE-toolkit\.venv\Lib\site-packages\tree_sitter\__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


Starting SATD classification...
Processing file 1/1000: PathPrefixPredicateTest.java
Processing file 2/1000: ApplicationTests.java
Processing file 3/1000: XForwardHeaderTest.java
Processing file 4/1000: AbstractExtensionTest.java
Processing file 5/1000: ListResultTest.java
Processing file 6/1000: MetadataOperatorTest.java
Processing file 7/1000: UnstructuredTest.java
Processing file 8/1000: ExtensionOperatorTest.java
Processing file 9/1000: ExtensionStoreUtilTest.java
Processing file 10/1000: DefaultSchemeWatcherManagerTest.java
Processing file 11/1000: FakeExtension.java
Processing file 12/1000: GroupVersionTest.java
Processing file 13/1000: JsonExtensionConverterTest.java
Processing file 14/1000: ComparatorsTest.java
Processing file 15/1000: RefTest.java
Processing file 16/1000: GroupVersionKindTest.java
Processing file 17/1000: DefaultSchemeManagerTest.java
Processing file 18/1000: ConfigMapTest.java
Processing file 19/1000: SchemeTest.java
Processing file 20/1000: JsonExtensionTest

KeyboardInterrupt: 

In [4]:
# print file 3333
print(ds[14]["file_name"])
print(ds[14]["content"])

RefTest.java
package run.halo.app.extension;

import static org.junit.jupiter.api.Assertions.assertFalse;
import static org.junit.jupiter.api.Assertions.assertTrue;
import static run.halo.app.extension.GroupVersionKind.fromAPIVersionAndKind;
import static run.halo.app.extension.GroupVersionKind.fromExtension;

import org.junit.jupiter.api.Test;

class RefTest {

    @Test
    void shouldHasSameGroupAndKind() {
        FakeExtension fake = new FakeExtension();
        Metadata metadata = new Metadata();
        metadata.setName("fake");
        fake.setMetadata(metadata);
        assertTrue(Ref.groupKindEquals(Ref.of(fake), fromExtension(fake.getClass())));
        // has different version
        assertTrue(Ref.groupKindEquals(Ref.of(fake),
            fromAPIVersionAndKind("fake.halo.run/v11111111111", "Fake")));
        assertFalse(Ref.groupKindEquals(Ref.of(fake),
            fromAPIVersionAndKind("fake.halo.run/v1alpha1", "NotFake")));
    }
}
